# Gold Layer: Dimensional Model (SkyLedger Airlines)

Builds the Gold layer for the SkyLedger Global Airline Analytics Lakehouse from the cleaned Silver tables (`skyledger_airlines.silver`) as the sole source of truth:

* `fact_bookings` — one row per flight-segment booking line, with all keys, measures, and attributes (passenger and airline geography kept distinct)
* `dim_passenger`, `dim_route`, `dim_airline`, `dim_geography` — one row per entity
* `vw_annual_booking_kpis` — annual net revenue, distinct bookings, active passengers, average booking value
* Reconciliation report — Gold vs Silver revenue and row counts, plus dimension key uniqueness validation

**Metric definitions (used consistently):** Net revenue = `ticket_price × (1 − discount)`; Bookings = `COUNT(DISTINCT booking_id)`; Active passengers = `COUNT(DISTINCT passenger)` in period; Average booking value = net revenue ÷ distinct bookings.

In [0]:
# Configuration — Gold layer build
# Silver tables (already cleaned and validated) are the sole source of truth.
CATALOG = "skyledger_airlines"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Silver source : {SILVER}")
print(f"Gold target   : {GOLD}")

In [0]:
# dim_geography — one row per nation, including its parent region.
# Shared by both hierarchies: passenger geography (via dim_passenger) and airline geography (via dim_airline).
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_geography AS
SELECT
  n.n_nationkey AS nation_key,
  n.n_name      AS nation_name,
  n.n_comment   AS nation_comment,
  r.r_regionkey AS region_key,
  r.r_name      AS region_name,
  r.r_comment   AS region_comment
FROM {SILVER}.nation n
JOIN {SILVER}.region r
  ON n.n_regionkey = r.r_regionkey
""")
display(spark.sql(f"SELECT * FROM {GOLD}.dim_geography"))

In [0]:
# dim_passenger — one row per passenger (passenger geography: nation + region of the passenger).
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_passenger AS
SELECT
  p.pa_passengerid   AS passenger_key,
  p.pa_name          AS passenger_name,
  p.pa_address       AS passenger_address,
  p.pa_phone         AS passenger_phone,
  p.pa_loyaltypoints AS loyalty_points,
  p.pa_travelpurpose AS travel_purpose,
  n.n_nationkey      AS passenger_nation_key,
  n.n_name           AS passenger_nation,
  n.n_regionkey      AS passenger_region_key,
  r.r_name           AS passenger_region
FROM {SILVER}.passenger p
JOIN {SILVER}.nation  n ON p.pa_nationkey = n.n_nationkey
JOIN {SILVER}.region  r ON n.n_regionkey = r.r_regionkey
""")
display(spark.table(f"{GOLD}.dim_passenger").limit(5))

In [0]:
# dim_route — one row per route (product master for each flight segment).
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_route AS
SELECT
  ro.ro_routeid      AS route_key,
  ro.ro_name         AS route_name,
  ro.ro_aircraft     AS aircraft,
  ro.ro_airline_brand AS airline_brand,
  ro.ro_routetype    AS route_type,
  ro.ro_capacity     AS route_capacity,
  ro.ro_cabinconfig  AS cabin_configuration,
  ro.ro_baseprice    AS base_price
FROM {SILVER}.route ro
""")
display(spark.table(f"{GOLD}.dim_route").limit(5))

In [0]:
# dim_airline — one row per airline (airline geography: nation + region of the carrier).
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_airline AS
SELECT
  a.al_airlineid AS airline_key,
  a.al_name      AS airline_name,
  a.al_address   AS airline_address,
  a.al_phone     AS airline_phone,
  a.al_rating    AS performance_rating,
  a.al_comment   AS airline_comment,
  n.n_nationkey  AS airline_nation_key,
  n.n_name       AS airline_nation,
  n.n_regionkey  AS airline_region_key,
  r.r_name       AS airline_region
FROM {SILVER}.airline a
JOIN {SILVER}.nation n ON a.al_nationkey = n.n_nationkey
JOIN {SILVER}.region r ON n.n_regionkey = r.r_regionkey
""")
display(spark.table(f"{GOLD}.dim_airline"))

In [0]:
# fact_bookings — one row per flight-segment booking line (booking_id + line_number is the grain).
# Keys and degenerate dimensions
#   - passenger geography: passenger_nation_* / passenger_region_* (where the passenger is from)
#   - airline geography:  airline_nation_* / airline_region_* (where the carrier is based)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_bookings AS
SELECT
  fb.fb_bookingid    AS booking_id,
  fb.fb_linenumber   AS line_number,
  b.bk_passengerid   AS passenger_key,
  fb.fb_routeid      AS route_key,
  fb.fb_airlineid    AS airline_key,

  -- Passenger geography (from the passenger's nation)
  pn.n_nationkey      AS passenger_nation_key,
  pn.n_name          AS passenger_nation,
  pr.r_regionkey     AS passenger_region_key,
  pr.r_name          AS passenger_region,

  -- Airline geography (from the carrier's nation)
  an.n_nationkey     AS airline_nation_key,
  an.n_name          AS airline_nation,
  ar.r_regionkey     AS airline_region_key,
  ar.r_name          AS airline_region,

  -- Measures
  fb.fb_seats        AS seats,
  fb.fb_ticketprice  AS ticket_price,
  fb.fb_discount     AS discount_rate,
  fb.fb_tax          AS tax_rate,
  CAST(fb.fb_ticketprice * (1 - fb.fb_discount) AS DECIMAL(18,2)) AS net_revenue,
  CAST(fb.fb_ticketprice * (1 - fb.fb_discount) * (1 + fb.fb_tax) AS DECIMAL(18,2)) AS net_revenue_with_tax,
  b.bk_totalprice    AS booking_total_price,

  -- Booking attributes
  b.bk_bookingdate   AS booking_date,
  b.bk_flightdate    AS flight_date,
  fb.fb_departuredate AS departure_date,
  fb.fb_arrivaldate  AS arrival_date,
  fb.fb_commitdate   AS commit_date,
  b.bk_status        AS booking_status,
  b.bk_bookingpriority AS booking_priority,
  b.bk_agent         AS booking_agent,
  fb.fb_segmentstatus AS segment_status,
  fb.fb_refundflag   AS refund_flag,
  fb.fb_cabinclass   AS cabin_class
FROM {SILVER}.flight_booking fb
JOIN {SILVER}.booking    b  ON fb.fb_bookingid  = b.bk_bookingid
JOIN {SILVER}.passenger p  ON b.bk_passengerid = p.pa_passengerid
JOIN {SILVER}.nation    pn ON p.pa_nationkey    = pn.n_nationkey
JOIN {SILVER}.region    pr ON pn.n_regionkey    = pr.r_regionkey
JOIN {SILVER}.airline   a  ON fb.fb_airlineid   = a.al_airlineid
JOIN {SILVER}.nation    an ON a.al_nationkey    = an.n_nationkey
JOIN {SILVER}.region    ar ON an.n_regionkey    = ar.r_regionkey
JOIN {SILVER}.route     ro ON fb.fb_routeid     = ro.ro_routeid
""")

row_cnt = spark.table(f"{GOLD}.fact_bookings").count()
print(f"fact_bookings rows: {row_cnt:,}")
display(spark.table(f"{GOLD}.fact_bookings").limit(5))

In [0]:
# vw_annual_booking_kpis — annual net revenue, distinct bookings, active passengers, average booking value.
# Metrics (per agreed definitions):
#   net_revenue         = SUM(ticket_price * (1 - discount))
#   total_bookings       = COUNT(DISTINCT booking_id)
#   active_passengers    = COUNT(DISTINCT passenger with >= 1 booking in the year)
#   average_booking_value = net_revenue / total_bookings
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_annual_booking_kpis AS
SELECT
  YEAR(booking_date) AS booking_year,
  CAST(SUM(net_revenue) AS DECIMAL(18,2)) AS net_revenue,
  COUNT(DISTINCT booking_id) AS total_bookings,
  COUNT(DISTINCT passenger_key) AS active_passengers,
  CAST(SUM(net_revenue) / COUNT(DISTINCT booking_id) AS DECIMAL(18,2)) AS average_booking_value
FROM {GOLD}.fact_bookings
GROUP BY YEAR(booking_date)
""")
display(spark.table(f"{GOLD}.vw_annual_booking_kpis"))

In [0]:
# Reconciliation & validation report
# 1) Gold fact total net revenue and row count vs an INDEPENDENT calculation from Silver.
# 2) Dimension key uniqueness checks.

silver_check = spark.sql(f"""
SELECT
  COUNT(*)                                    AS silver_rows,
  CAST(SUM(fb_ticketprice * (1 - fb_discount)) AS DECIMAL(18,2)) AS silver_net_revenue
FROM {SILVER}.flight_booking
""").collect()[0]

gold_check = spark.sql(f"""
SELECT
  COUNT(*)            AS gold_rows,
  SUM(net_revenue)    AS gold_net_revenue
FROM {GOLD}.fact_bookings
""").collect()[0]

row_diff = gold_check["gold_rows"] - silver_check["silver_rows"]
rev_diff = float(gold_check["gold_net_revenue"] or 0) - float(silver_check["silver_net_revenue"] or 0)

report = []
report.append((
    "Gold vs Silver — row count",
    f"Gold={gold_check['gold_rows']:,} | Silver={silver_check['silver_rows']:,}",
    float(row_diff),
    "PASS" if row_diff == 0 else "FAIL",
))
report.append((
    "Gold vs Silver — net revenue",
    f"Gold={gold_check['gold_net_revenue']:,.2f} | Silver={silver_check['silver_net_revenue']:,.2f}",
    round(rev_diff, 2),
    "PASS" if abs(rev_diff) < 0.01 else "FAIL",
))

# Dimension key uniqueness (and fact grain uniqueness)
uniqueness_specs = [
    ("dim_geography", "nation_key"),
    ("dim_passenger", "passenger_key"),
    ("dim_route",     "route_key"),
    ("dim_airline",   "airline_key"),
    ("fact_bookings", "booking_id, line_number"),
]
for table, key in uniqueness_specs:
    res = spark.sql(f"""
        SELECT COUNT(*) AS total_rows, COUNT(DISTINCT {key}) AS distinct_keys
        FROM {GOLD}.{table}
        """).collect()[0]
    dup = res["total_rows"] - res["distinct_keys"]
    report.append((
        f"Key uniqueness — {table} ({key})",
        f"rows={res['total_rows']:,} | distinct keys={res['distinct_keys']:,}",
        float(dup),
        "PASS" if dup == 0 else "FAIL",
    ))

from pyspark.sql import Row
reconciliation_report = spark.createDataFrame(
    [Row(check= c, detail=d, difference=diff, status=s) for c, d, diff, s in report]
)
print("=" * 70)
print("GOLD LAYER RECONCILIATION REPORT — SkyLedger Airlines (TPC-A practice data)")
print("=" * 70)
display(reconciliation_report)

overall = "PASS" if all(r[3] == "PASS" for r in report) else "FAIL"
print(f"\nOVERALL STATUS: {overall}")